# Acarí / Lomas 2018 — preparación del dataset externo congelado

Este cuaderno ejecuta **únicamente** la reconstrucción de variables previa a la validación predictiva:

- inventario y QC de 20 registros CEOIS/CISMID recuperables;
- recomputación de RotD50 a T = 1.0 s, 5 % de amortiguamiento;
- asignación de Vs30 desde el archivo congelado `global_vs30.grd`;
- descarga de la geometría de ruptura finita oficial de USGS y cálculo 3D de Rrup;
- ensamblaje del dataset final y generación de SHA-256.

**Este cuaderno NO carga ni ejecuta V5.1.** El modelo solo podrá ejecutarse después de verificar el archivo `FINAL_FROZEN` y su manifiesto.

Reglas ya fijadas antes de V5.1: LIM025 se incluye si pasa el mismo QC; JUN001 se documenta como no disponible y no se reconstruye; Mw para el modelo se fija en **USGS Mww = 7.1** para el evento `us2000cjfy`.

**Corrección QC v1.1:** `START_TIME` se valida numéricamente con tolerancia máxima de 10 µs; no se exige igualdad textual exacta. No altera datos ni RotD50.


In [ ]:
# 1) Dependencias y montaje de Google Drive
import os, re, io, json, math, hashlib, shutil, sys, subprocess
from pathlib import Path

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, name in [('numpy','numpy'),('pandas','pandas'),('xarray','xarray'),('netCDF4','netCDF4'),('pyproj','pyproj'),('requests','requests')]:
    ensure(pkg, name)

import numpy as np
import pandas as pd
import xarray as xr
import requests
from pyproj import Transformer

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/Tesis/Datos/Validacion_Externa/Acari2018')
RAW = BASE / 'raw_ceois'
QC = BASE / 'QC'
DATOS = Path('/content/drive/MyDrive/Tesis/Datos')
GRID_DRIVE = DATOS / 'global_vs30.grd'
VS30_REF = DATOS / 'Vs30_estaciones_2025_0396.csv'

QC.mkdir(parents=True, exist_ok=True)
assert RAW.exists(), f'No existe {RAW}'
assert GRID_DRIVE.exists(), f'No existe {GRID_DRIVE}'
print('BASE:', BASE)
print('GRID:', GRID_DRIVE, f'({GRID_DRIVE.stat().st_size/1024**2:.1f} MiB)')

In [ ]:
# 2) Reglas congeladas y referencias de auditoría
EVENT_ID = 'us2000cjfy'
MW_MODEL = 7.1
G0 = 980.665  # cm/s^2
T = 1.0
XI = 0.05
BETA = 1/4
GAMMA = 1/2

EXPECTED_ROTD50 = {
'CISMID_CM_ANCON.txt':0.001893667325,
'CISMID_CM_CARAB.txt':0.002320123803,
'CISMID_CM_CENEP.txt':0.003479276159,
'CISMID_CM_CEPRE.txt':0.002981935737,
'CISMID_CM_CIPCN.txt':0.004601219737,
'CISMID_CM_COMAS.txt':0.002320510223,
'CISMID_CM_DHNPE.txt':0.010120902456,
'CISMID_CM_IMCA.txt':0.004270367237,
'CISMID_CM_INDEP.txt':0.001953619844,
'CISMID_CM_INICT.txt':0.002910053233,
'CISMID_CM_JALVA.txt':0.001549512714,
'CISMID_CM_MDPP.txt':0.004624944789,
'CISMID_CM_OLIVO.txt':0.003176184760,
'CISMID_CM_SROSA.txt':0.002014008231,
'CISMID_CM_UNFV.txt':0.004703445093,
'CISMID_CM_USMP.txt':0.002114046981,
'CISMID_SC_SCARQ.txt':0.018899820044,
'CISMID_SC_SCCUS.txt':0.015865744525,
'CISMID_SC_SCICA.txt':0.084795487274,
'CISMID_SC_SCTAC.txt':0.003918944342,
}

EXPECTED_FILES = sorted(EXPECTED_ROTD50)
print('Registros esperados:', len(EXPECTED_FILES))
print('Mw fijada para el modelo:', MW_MODEL)

In [ ]:
# 3) Funciones de lectura, hash, QC y RotD50
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def parse_ceois_txt(path):
    text = Path(path).read_text(errors='replace')
    lines = text.splitlines()

    def grab(prefix):
        for line in lines:
            if line.startswith(prefix):
                return line.split(':',1)[1].strip()
        raise ValueError(f'No se encontró {prefix} en {path}')

    station_field = grab('# STATION')
    m = re.match(r'([^()]+?)\s*\(([^ )]+)', station_field)
    station = m.group(1).strip() if m else station_field.split('(')[0].strip()
    code_full = m.group(2).strip() if m else ''
    code = code_full.split('-')[0] if code_full else station
    if station == 'SCICA':
        code_historical = 'ICA004'
    else:
        code_historical = code

    channels = grab('# CHANNEL').split()
    fs = float(grab('# SAMPLING FREQUENCY (Hz)'))
    lat_s, lon_s = grab('# COORDINATES').split(',')
    lat, lon = float(lat_s), float(lon_s)
    date = grab('# DATE')
    origin_local = grab('# ORIGIN TIME (Local)')
    mw_header = float(grab('# MAGNITUDE').split()[0])
    start_time = grab('# START_TIME (UTC-0)')
    n_decl = int(grab('# NUMBER DATA'))
    units = grab('# DATA UNITS')
    pga_decl = np.array([float(x) for x in grab('# MAXIMUM ACCELERATION').split()])

    idx = next(i for i,l in enumerate(lines) if l.startswith('# 5. ACCELERATION DATA'))
    data_lines = []
    for l in lines[idx+1:]:
        s = l.strip()
        if not s or s.startswith('#'):
            continue
        data_lines.append(s)
    data = np.loadtxt(io.StringIO('\n'.join(data_lines)))
    if data.ndim == 1:
        data = data.reshape(-1, len(channels))

    assert data.shape == (n_decl, 3), (path, data.shape, n_decl)
    assert np.isfinite(data).all(), f'NaN/Inf: {path}'
    assert abs(fs - 200.0) < 1e-12, f'Fs inesperada: {path}'
    assert units.replace('²','2').replace('^2','2').lower() in ['cm/s2','cm/s2'], f'Unidades inesperadas: {units}'
    assert len(channels) == 3
    # Dos horizontales + vertical según nomenclatura CEOIS usada en el evento.
    assert channels[0].endswith('E') and channels[1].endswith('N') and channels[2].endswith('Z'), (path,channels)

    # PGA de cabecera contra extremos observados, admitiendo el redondeo publicado.
    extrema = np.array([data[:,j][np.argmax(np.abs(data[:,j]))] for j in range(3)])
    assert np.allclose(extrema, pga_decl, atol=5e-3, rtol=0), (path, extrema, pga_decl)

    # QC grueso: no flatline/ceros por >=1 s.
    min_run = int(fs)
    for j in range(3):
        x = data[:,j]
        eq = np.isclose(np.diff(x), 0.0, atol=0, rtol=0)
        maxrun = 0; cur = 0
        for flag in eq:
            cur = cur + 1 if flag else 0
            maxrun = max(maxrun, cur)
        assert maxrun < min_run, f'Flatline >=1 s en {path} canal {j}'

    return {
        'file':Path(path).name, 'station':station, 'code':code, 'code_full':code_full,
        'code_historical':code_historical, 'channels':' '.join(channels), 'fs_hz':fs,
        'lat':lat, 'lon':lon, 'date':date, 'origin_local':origin_local,
        'mw_ceois_header':mw_header, 'start_time':start_time, 'n_samples':n_decl,
        'units':units, 'pga_e':pga_decl[0], 'pga_n':pga_decl[1], 'pga_z':pga_decl[2],
        'raw_sha256':sha256_file(path), 'data':data
    }

def sdof_newmark_relative_displacement(ag_cm_s2, dt, T=1.0, zeta=0.05, beta=0.25, gamma=0.5):
    ag = np.asarray(ag_cm_s2, dtype=float)
    n = ag.size
    w = 2*np.pi/T
    m = 1.0
    c = 2*zeta*w*m
    k = w*w*m
    p = -ag

    u = np.zeros(n)
    v = np.zeros(n)
    acc = np.zeros(n)
    acc[0] = (p[0] - c*v[0] - k*u[0])/m

    a0 = 1/(beta*dt*dt)
    a1 = gamma/(beta*dt)
    a2 = 1/(beta*dt)
    a3 = 1/(2*beta)-1
    a4 = gamma/beta-1
    a5 = dt*(gamma/(2*beta)-1)
    khat = k + a0*m + a1*c

    for i in range(1,n):
        phat = p[i] + m*(a0*u[i-1] + a2*v[i-1] + a3*acc[i-1]) \
                       + c*(a1*u[i-1] + a4*v[i-1] + a5*acc[i-1])
        u[i] = phat/khat
        acc[i] = a0*(u[i]-u[i-1]) - a2*v[i-1] - a3*acc[i-1]
        v[i] = v[i-1] + dt*((1-gamma)*acc[i-1] + gamma*acc[i])
    return u

def rotd50_t1_g(h1, h2, fs):
    # Regla congelada: segmento común (ya igual), sustracción de media, Newmark, rotaciones 0–179°.
    h1 = np.asarray(h1,float) - np.mean(h1)
    h2 = np.asarray(h2,float) - np.mean(h2)
    dt = 1.0/fs
    u1 = sdof_newmark_relative_displacement(h1, dt, T=T, zeta=XI, beta=BETA, gamma=GAMMA)
    u2 = sdof_newmark_relative_displacement(h2, dt, T=T, zeta=XI, beta=BETA, gamma=GAMMA)
    w2 = (2*np.pi/T)**2
    vals = []
    for deg in range(180):
        th = np.deg2rad(deg)
        ur = u1*np.cos(th) + u2*np.sin(th)
        psa = w2*np.max(np.abs(ur))/G0
        vals.append(psa)
    return float(np.median(vals))

In [ ]:
# 4) QC de los 20 registros y recomputación independiente de RotD50
actual_files = sorted(p.name for p in RAW.glob('*.txt'))
missing = sorted(set(EXPECTED_FILES)-set(actual_files))
extra = sorted(set(actual_files)-set(EXPECTED_FILES))
assert not missing, f'Faltan archivos: {missing}'
assert not extra, f'Hay archivos no predeclarados en raw_ceois: {extra}'
assert len(actual_files) == 20

records=[]
for name in actual_files:
    r = parse_ceois_txt(RAW/name)
    rot = rotd50_t1_g(r['data'][:,0], r['data'][:,1], r['fs_hz'])
    r['RotD50_g'] = rot
    ref = EXPECTED_ROTD50[name]
    r['RotD50_ref_g'] = ref
    r['RotD50_rel_diff'] = abs(rot-ref)/ref
    assert np.isclose(rot, ref, rtol=2e-6, atol=2e-10), f'RotD50 no reproduce QC previo: {name}: {rot} vs {ref}'
    records.append(r)

# Ventana común / metadatos homogéneos
assert len({r['n_samples'] for r in records}) == 1
assert len({r['fs_hz'] for r in records}) == 1
assert len({r['date'] for r in records}) == 1

# START_TIME no se compara como texto exacto: CEOIS puede representar el mismo instante
# con diferencias de redondeo del orden de microsegundos. A 200 Hz, dt = 0.005 s.
# Para no ocultar una desalineación real, se admite únicamente una dispersión <= 10 us.
start_times = pd.to_datetime([r['start_time'] for r in records], utc=True)
start_min = start_times.min()
start_max = start_times.max()
start_spread_s = (start_max - start_min).total_seconds()
START_TOL_S = 10e-6
print('START_TIME mínimo:', start_min)
print('START_TIME máximo:', start_max)
print('Dispersión START_TIME:', start_spread_s*1e6, 'microsegundos')
print('Periodo de muestreo:', 1.0/records[0]['fs_hz'], 's')
assert start_spread_s <= START_TOL_S, (
    f'Diferencia real de START_TIME mayor que {START_TOL_S*1e6:.0f} us: '
    f'{start_spread_s*1e6:.3f} us. Revisar antes de continuar.'
)

qc_df = pd.DataFrame([{k:v for k,v in r.items() if k!='data'} for r in records])
qc_path = QC/'Acari2018_senales_QC_RotD50.csv'
qc_df.to_csv(qc_path,index=False,float_format='%.12g',lineterminator='\n')
print(qc_df[['file','station','code','lat','lon','RotD50_g','RotD50_rel_diff']].to_string(index=False))
print('\nQC señales: 20/20 PASS')
print('Máx. diferencia relativa RotD50 vs auditoría previa:', qc_df.RotD50_rel_diff.max())
print('Guardado:', qc_path)

In [ ]:
# 5) Vs30: copia local del grid congelado y detección automática de ejes/variable
LOCAL_GRID = Path('/content/global_vs30.grd')
if not LOCAL_GRID.exists() or LOCAL_GRID.stat().st_size != GRID_DRIVE.stat().st_size:
    print('Copiando grid de Drive a disco local para lectura eficiente...')
    shutil.copy2(GRID_DRIVE, LOCAL_GRID)

grid_sha256 = sha256_file(LOCAL_GRID)
print('SHA256 global_vs30.grd:', grid_sha256)

ds = xr.open_dataset(LOCAL_GRID)
print(ds)

# Elegir la variable 2D principal.
vars_2d = [v for v in ds.data_vars if ds[v].ndim >= 2]
assert vars_2d, 'No se detectó variable 2D en global_vs30.grd'
# Preferencias típicas de GMT/USGS.
pref_vars = [v for v in vars_2d if v.lower() in ('z','vs30','mosaic')]
var_name = pref_vars[0] if pref_vars else vars_2d[0]
da = ds[var_name].squeeze()

dims = list(da.dims)
def pick_dim(kind):
    if kind=='lon':
        candidates = [d for d in dims if d.lower() in ('lon','longitude','x') or 'lon' in d.lower()]
    else:
        candidates = [d for d in dims if d.lower() in ('lat','latitude','y') or 'lat' in d.lower()]
    return candidates[0] if candidates else None
lon_dim, lat_dim = pick_dim('lon'), pick_dim('lat')
assert lon_dim and lat_dim and lon_dim != lat_dim, f'No se identificaron ejes lat/lon: {dims}'
print('Variable:', var_name, '| lon_dim:', lon_dim, '| lat_dim:', lat_dim)

lon_vals = np.asarray(ds[lon_dim].values)
lat_vals = np.asarray(ds[lat_dim].values)
lon_0360 = np.nanmin(lon_vals) >= 0 and np.nanmax(lon_vals) > 180

def vs30_nearest(lat, lon):
    qlon = lon + 360 if lon_0360 and lon < 0 else lon
    out = da.sel({lat_dim:lat, lon_dim:qlon}, method='nearest')
    val = float(np.asarray(out.values).squeeze())
    # coordenada realmente usada
    used_lat = float(np.asarray(out[lat_dim].values).squeeze())
    used_lon = float(np.asarray(out[lon_dim].values).squeeze())
    if lon_0360 and used_lon > 180:
        used_lon -= 360
    assert np.isfinite(val) and val > 0, (lat,lon,val)
    return val, used_lat, used_lon

In [ ]:
# 6) Auditoría de la regla Vs30 contra un archivo previo generado con el mismo grid + xarray
assert VS30_REF.exists(), f'Falta referencia de auditoría: {VS30_REF}'
ref_df = pd.read_csv(VS30_REF)
audit=[]
for _,row in ref_df.iterrows():
    val, glat, glon = vs30_nearest(float(row['lat']), float(row['lon']))
    audit.append({
        'code':row['code'],'lat':row['lat'],'lon':row['lon'],
        'Vs30_ref':row['Vs30_USGS_m_s'],'Vs30_recalc':val,
        'abs_diff':abs(val-float(row['Vs30_USGS_m_s'])),
        'grid_lat':glat,'grid_lon':glon
    })
audit_df = pd.DataFrame(audit)
maxdiff = audit_df.abs_diff.max()
print(audit_df.to_string(index=False))
print('Máx. diferencia absoluta Vs30 contra referencia:', maxdiff)
assert maxdiff < 1e-3, 'La asignación Vs30 no reproduce la referencia del pipeline anterior.'
audit_df.to_csv(QC/'Acari2018_auditoria_metodo_Vs30.csv',index=False,float_format='%.12g',lineterminator='\n')

vsrows=[]
for r in records:
    v, glat, glon = vs30_nearest(r['lat'],r['lon'])
    vsrows.append({'file':r['file'],'station':r['station'],'code':r['code'],'lat':r['lat'],'lon':r['lon'],
                   'Vs30_m_s':v,'grid_lat':glat,'grid_lon':glon,'method':'nearest xarray; global_vs30.grd'})
vs_df = pd.DataFrame(vsrows)
vs_df.to_csv(QC/'Acari2018_Vs30_auditoria.csv',index=False,float_format='%.12g',lineterminator='\n')
print('\nVs30 Acarí 2018:')
print(vs_df.to_string(index=False))

In [ ]:
# 7) USGS: detalle del evento y selección determinista de ruptura finita ShakeMap
DETAIL_URL = f'https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/{EVENT_ID}.geojson'
resp = requests.get(DETAIL_URL, timeout=30)
resp.raise_for_status()
detail = resp.json()
(QC/'Acari2018_USGS_event_detail.json').write_text(json.dumps(detail,indent=2))

print('USGS id:', detail.get('id'))
print('USGS mag:', detail['properties'].get('mag'), detail['properties'].get('magType'))
print('Hipocentro:', detail['geometry']['coordinates'])
assert detail.get('id') == EVENT_ID
assert abs(float(detail['properties']['mag']) - MW_MODEL) < 0.05, 'Mw USGS no coincide con la Mw predeclarada 7.1'

products = detail['properties'].get('products',{})
shakemaps = products.get('shakemap',[])
assert shakemaps, 'USGS detail no contiene producto shakemap.'

# Orden predeclarado: mayor preferredWeight y, en empate, actualización más reciente.
shakemaps_sorted = sorted(shakemaps, key=lambda p:(p.get('preferredWeight',0),p.get('updateTime',0)), reverse=True)
selected_sm = None
rupture_key = None
for p in shakemaps_sorted:
    keys = list(p.get('contents',{}).keys())
    rk = [k for k in keys if k.lower().endswith('rupture.json') or 'rupture.json' in k.lower()]
    if rk:
        selected_sm = p
        rupture_key = sorted(rk, key=lambda x:(0 if x.lower()=='download/rupture.json' else 1, len(x)))[0]
        break

if selected_sm is None:
    # No inventar geometría. Dejar trazabilidad y detener antes de Rrup.
    available = {p.get('id','?'): sorted(p.get('contents',{}).keys()) for p in shakemaps_sorted}
    (QC/'Acari2018_USGS_shakemap_contents_sin_rupture.json').write_text(json.dumps(available,indent=2))
    raise RuntimeError('No se encontró rupture.json en productos ShakeMap USGS. Se detiene sin estimar Rrup ni ejecutar V5.1.')

rupture_url = selected_sm['contents'][rupture_key]['url']
print('ShakeMap seleccionado:', selected_sm.get('id'))
print('preferredWeight:', selected_sm.get('preferredWeight'),'updateTime:',selected_sm.get('updateTime'))
print('Rupture key:', rupture_key)
print('Rupture URL:', rupture_url)

rr = requests.get(rupture_url, timeout=30)
rr.raise_for_status()
rupture = rr.json()
rupture_path = QC/'Acari2018_USGS_rupture.json'
rupture_path.write_text(json.dumps(rupture,indent=2))
rupture_sha256 = sha256_file(rupture_path)
print('SHA256 rupture.json:', rupture_sha256)

In [ ]:
# 8) Extracción de polígonos 3D y cálculo geométrico de Rrup
def collect_polygon_rings(obj):
    rings=[]
    def geom(g):
        if not g:
            return
        typ = g.get('type')
        c = g.get('coordinates')
        if typ == 'Polygon':
            if c:
                rings.append(c[0])  # anillo exterior
        elif typ == 'MultiPolygon':
            for poly in c or []:
                if poly:
                    rings.append(poly[0])
        elif typ == 'GeometryCollection':
            for gg in g.get('geometries',[]):
                geom(gg)
    typ=obj.get('type') if isinstance(obj,dict) else None
    if typ=='FeatureCollection':
        for f in obj.get('features',[]):
            geom(f.get('geometry'))
    elif typ=='Feature':
        geom(obj.get('geometry'))
    elif typ in ('Polygon','MultiPolygon','GeometryCollection'):
        geom(obj)
    else:
        # Algunos rupture.json contienen un objeto con key geometry.
        if isinstance(obj,dict) and 'geometry' in obj:
            geom(obj['geometry'])
    return rings

rings_raw = collect_polygon_rings(rupture)
assert rings_raw, 'No se pudo extraer Polygon/MultiPolygon del rupture.json'

rings=[]
for ring in rings_raw:
    arr=np.asarray(ring,dtype=float)
    if arr.shape[0] >= 2 and np.allclose(arr[0,:2],arr[-1,:2]):
        arr=arr[:-1]
    assert arr.ndim==2 and arr.shape[1]>=3 and arr.shape[0]>=3, f'Geometría sin profundidad 3D: {arr.shape}'
    assert np.isfinite(arr[:,:3]).all()
    rings.append(arr[:,:3])
print('Polígonos 3D:',len(rings),'| vértices únicos por polígono:',[len(r) for r in rings])
print('Rango de profundidades (km):', min(r[:,2].min() for r in rings), 'a', max(r[:,2].max() for r in rings))

# WGS84 geodésico 3D -> ECEF. Profundidad positiva hacia abajo => altura elipsoidal negativa.
tf = Transformer.from_crs('EPSG:4979','EPSG:4978',always_xy=True)

def ecef(lon,lat,depth_km=0.0):
    x,y,z=tf.transform(float(lon),float(lat),-1000.0*float(depth_km))
    return np.array([x,y,z],dtype=float)

def point_triangle_distance(p,a,b,c):
    # Distancia mínima punto-triángulo (Ericson, Real-Time Collision Detection).
    ab=b-a; ac=c-a; ap=p-a
    d1=np.dot(ab,ap); d2=np.dot(ac,ap)
    if d1<=0 and d2<=0: return np.linalg.norm(ap)
    bp=p-b; d3=np.dot(ab,bp); d4=np.dot(ac,bp)
    if d3>=0 and d4<=d3: return np.linalg.norm(bp)
    vc=d1*d4-d3*d2
    if vc<=0 and d1>=0 and d3<=0:
        v=d1/(d1-d3); proj=a+v*ab; return np.linalg.norm(p-proj)
    cp=p-c; d5=np.dot(ab,cp); d6=np.dot(ac,cp)
    if d6>=0 and d5<=d6: return np.linalg.norm(cp)
    vb=d5*d2-d1*d6
    if vb<=0 and d2>=0 and d6<=0:
        w=d2/(d2-d6); proj=a+w*ac; return np.linalg.norm(p-proj)
    va=d3*d6-d5*d4
    if va<=0 and (d4-d3)>=0 and (d5-d6)>=0:
        w=(d4-d3)/((d4-d3)+(d5-d6)); proj=b+w*(c-b); return np.linalg.norm(p-proj)
    denom=1.0/(va+vb+vc)
    v=vb*denom; w=vc*denom
    proj=a+ab*v+ac*w
    return np.linalg.norm(p-proj)

triangles=[]
for ring in rings:
    pts=[ecef(lon,lat,dep) for lon,lat,dep in ring]
    # Fan triangulation del polígono; para las rupturas ShakeMap usuales cada anillo es cuadrilateral.
    for i in range(1,len(pts)-1):
        triangles.append((pts[0],pts[i],pts[i+1]))
assert triangles

def rrup_km(site_lat,site_lon):
    p=ecef(site_lon,site_lat,0.0)
    return min(point_triangle_distance(p,*tri) for tri in triangles)/1000.0

rrrows=[]
for r in records:
    d=rrup_km(r['lat'],r['lon'])
    assert np.isfinite(d) and d>0 and d<2000, (r['code'],d)
    rrrows.append({'file':r['file'],'station':r['station'],'code':r['code'],'lat':r['lat'],'lon':r['lon'],'Rrup_km':d})
rr_df=pd.DataFrame(rrrows)
rr_df.to_csv(QC/'Acari2018_Rrup_auditoria.csv',index=False,float_format='%.12g',lineterminator='\n')
print(rr_df.to_string(index=False))

In [ ]:
# 9) Ensamble, congelamiento y manifiesto SHA-256 — TODAVÍA SIN V5.1
base_df = qc_df[['file','station','code','code_full','code_historical','lat','lon','mw_ceois_header','RotD50_g','raw_sha256']].copy()
final = base_df.merge(vs_df[['file','Vs30_m_s','grid_lat','grid_lon']],on='file',validate='one_to_one')
final = final.merge(rr_df[['file','Rrup_km']],on='file',validate='one_to_one')
final.insert(7,'Mw',MW_MODEL)
final['ln_Sa_obs'] = np.log(final['RotD50_g'])
final['Mw_source'] = f'USGS {EVENT_ID} Mww'
final['Rrup_source'] = f'USGS ShakeMap rupture.json ({selected_sm.get("id")})'
final['Vs30_source'] = 'USGS global_vs30.grd; nearest xarray'
final['Sa_definition'] = 'RotD50 PSA T=1.0s xi=5%; g0=980.665 cm/s2'
final['signal_source'] = 'CEOIS/CISMID sin filtrado frecuencial; baseline corrected por proveedor'

# Orden determinista independiente de desempeño.
final = final.sort_values(['code','file']).reset_index(drop=True)
assert len(final)==20
assert final[['Mw','Rrup_km','Vs30_m_s','RotD50_g','ln_Sa_obs']].notna().all().all()
assert (final['RotD50_g']>0).all() and (final['Vs30_m_s']>0).all() and (final['Rrup_km']>0).all()

frozen_path = BASE/'Acari2018_20_estaciones_Rrup_Vs30_RotD50_FINAL_FROZEN.csv'
final.to_csv(frozen_path,index=False,float_format='%.12g',lineterminator='\n')
dataset_sha256 = sha256_file(frozen_path)

manifest = {
    'event':'Acari/Lomas 2018',
    'event_id_usgs':EVENT_ID,
    'Mw_model':MW_MODEL,
    'Mw_source':f'USGS {EVENT_ID} Mww',
    'n_stations_included':20,
    'jun001_status':'reportada históricamente; no disponible actualmente en CEOIS; no reconstruida',
    'lim025_rule':'incluida por disponibilidad actual en CEOIS + mismo QC, decisión fijada antes de V5.1',
    'signal_processing':'common segment; mean subtraction; Newmark beta=1/4 gamma=1/2; PSA T=1.0s xi=5%; rotations 0-179 deg; RotD50; g0=980.665 cm/s2',
    'signal_provenance':'CEOIS/CISMID records without frequency filtering; provider labels files BASELINE CORRECTED',
    'vs30_file':str(GRID_DRIVE),
    'vs30_grid_sha256':grid_sha256,
    'vs30_method':'nearest xarray; audited against Vs30_estaciones_2025_0396.csv',
    'usgs_detail_url':DETAIL_URL,
    'shakemap_product_id':selected_sm.get('id'),
    'shakemap_preferredWeight':selected_sm.get('preferredWeight'),
    'shakemap_updateTime':selected_sm.get('updateTime'),
    'rupture_content_key':rupture_key,
    'rupture_url':rupture_url,
    'rupture_sha256':rupture_sha256,
    'rrup_method':'minimum 3D ECEF point-to-triangle distance to USGS rupture polygon(s), fan triangulated',
    'dataset_file':frozen_path.name,
    'dataset_sha256':dataset_sha256,
    'raw_file_sha256':{r['file']:r['raw_sha256'] for r in records},
    'model_v5_1_executed':False
}
manifest_path = BASE/'Acari2018_FREEZE_MANIFEST.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False))

# Archivo de lectura rápida para auditoría.
print(final[['code','station','lat','lon','Mw','Rrup_km','Vs30_m_s','RotD50_g','ln_Sa_obs']].to_string(index=False))
print('\n=== CONGELAMIENTO COMPLETADO ===')
print('Dataset:', frozen_path)
print('SHA256 dataset:', dataset_sha256)
print('Manifest:', manifest_path)
print('SHA256 rupture:', rupture_sha256)
print('SHA256 global_vs30.grd:', grid_sha256)
print('\nV5.1 EJECUTADO: NO')

## Criterio de salida

El cuaderno debe terminar mostrando **“V5.1 EJECUTADO: NO”** y un SHA-256 para el dataset congelado.

Si alguna validación falla, el cuaderno se detiene antes del congelamiento. No se debe modificar una regla, eliminar una estación o cambiar una fuente en respuesta a un futuro resultado del modelo.